# Fast buoy detector: zone tiles + synthetic buoys (YOLO11n, 640 px)

Trains a tiny detector **only on the zone near the skier** (LEFT / MIDDLE / RIGHT tiles of the water band), with
synthetic buoys in 6 colours and many sizes, spray occlusion, blur and noise. Then tests it on the hand-checked
buoys of videos it never saw, next to the zero-shot models.

**Colab:** Runtime → Change runtime type → any GPU (T4 is enough) → Run all. ~10–15 min in total.
**Locally:** same commands in a terminal, see `docs/BUOY_TRAINING.md` (CPU timing is there too).


In [ ]:
EPOCHS = 30
import os, multiprocessing; WORKERS = min(8, multiprocessing.cpu_count())
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None
url = f"https://{tok}@github.com/lytvynenkomaksim/CV_test.git" if tok else "https://github.com/lytvynenkomaksim/CV_test.git"
if not os.path.exists('/content/CV_test'):
    !git clone --depth 1 -b main {url} /content/CV_test
%cd /content/CV_test
!pip install -q ultralytics tqdm
!mkdir -p models && cd models && [ -f yolo11n.pt ] || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt

## 1. Data: frames from the repo videos → zone tiles + synthetic buoys (progress bars)

In [ ]:
!python tools/extract_label_frames.py --dataset data/buoy_dataset_v2
!python tools/make_zone_dataset.py --synth-per-tile 1
!cat data/buoy_zone/stats.json

## 2. Train (validates every epoch)

In [ ]:
!python tools/train_buoy_yolo.py --data data/buoy_zone/data.yaml --model yolo11n.pt --imgsz 640 \
    --epochs {EPOCHS} --patience 8 --batch 64 --workers {WORKERS} --cache ram \
    --name buoy_zone_yolo11n --out buoy_zone_yolo11n.pt

## 3. Test on hand-checked buoys (held-out videos), next to the zero-shot models

In [ ]:
!python tools/zeroshot_buoys.py --models yolo=buoy_zone_yolo11n.pt@1280 --out results/zeroshot
!python tools/zeroshot_score.py --preds results/zeroshot/preds --out results/zeroshot
from IPython.display import Markdown; Markdown(open('results/zeroshot/summary.md').read())

In [ ]:
# download the weights + results
!zip -qr buoy_zone_result.zip models/buoy_zone_yolo11n.pt models/buoy_zone_yolo11n.onnx runs/train/buoy_zone_yolo11n results/zeroshot
from google.colab import files; files.download('buoy_zone_result.zip')